In [3]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime as dt
from pathlib import Path
import os
from tqdm import tqdm
from fnmatch import fnmatch
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.spatial.distance import cdist
import datacube
from datacube.drivers.netcdf import write_dataset_to_netcdf

In [4]:
#Function to select certain variables from files and concatenate
def process_daily_files(result_dir, files_to_open, variables_to_select, progress_bar=None):
    # Open the daily files as an xarray dataset
    ds = xr.open_mfdataset(str(result_dir / files_to_open), combine='nested')

    # Select specific variables
    ds = ds[variables_to_select]

    # Update progress bar
    if progress_bar:
        progress_bar.update(1)
    return ds

# Function to calculate Haversine distance from observed monitoring sites and irregularly gridded cells
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # approximate radius of Earth in km
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    distance = R * c
    return distance

In [5]:
result_dir = Path('/glade/derecho/scratch/demurray/archive/MUSICA_FCnudged30x8_20172018_082024/atm/hist')

files_to_open = 'MUSICA_FCnudged30x8_20172018_082024.cam.h0.*.nc'

variables_to_select = ['lat', 'lon', 'area','pom_a1SFWET', 'pom_a4SFWET', 'pom_c1SFWET', 'pom_c4SFWET', 
                       'soa1_a1SFWET', 'soa1_c1SFWET', 'soa1_a2SFWET', 'soa1_c2SFWET', 
                       'soa2_a1SFWET', 'soa2_c1SFWET', 'soa2_a2SFWET', 'soa2_c2SFWET', 
                       'soa3_a1SFWET', 'soa3_c1SFWET', 'soa3_a2SFWET',  'soa3_c2SFWET', 
                       'soa4_a1SFWET', 'soa4_c1SFWET', 'soa4_a2SFWET',  'soa4_c2SFWET', 
                       'soa5_a1SFWET', 'soa5_c1SFWET', 'soa5_a2SFWET',  'soa5_c2SFWET', 
                       'bc_a1SFWET', 'bc_a4SFWET', 'bc_c1SFWET', 'bc_c4SFWET', 'so4_a1SFWET', 
                       'so4_a2SFWET', 'so4_a3SFWET', 'so4_c1SFWET', 'so4_c2SFWET', 'so4_c3SFWET',
                       'WD_NH4', 'WD_NH3', 'WD_HNO3', 'PRECC', 'PRECL']

files_list = list(result_dir.glob(files_to_open))

# Initialize tqdm progress bar
progress = tqdm(total=len(files_list))

processed_data = []
for file in files_list:
    processed_data.append(process_daily_files(result_dir, file.relative_to(result_dir), variables_to_select, progress_bar=progress))

progress.close()

musica_daily = xr.concat(processed_data, dim='time').sortby('time')
musica_daily

100%|██████████| 729/729 [03:21<00:00,  3.63it/s]


<xarray.Dataset>
Dimensions:       (time: 729, ncol: 174098)
Coordinates:
  * time          (time) datetime64[ns] 2017-01-01 2017-01-02 ... 2018-12-30
Dimensions without coordinates: ncol
Data variables: (12/42)
    lat           (time, ncol) float64 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    lon           (time, ncol) float64 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    area          (time, ncol) float64 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    pom_a1SFWET   (time, ncol) float32 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    pom_a4SFWET   (time, ncol) float32 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    pom_c1SFWET   (time, ncol) float32 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    ...            ...
    so4_c3SFWET   (time, ncol) float32 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    WD_NH4        (time, ncol) float32 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    WD_NH3        (time, ncol) float32 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    WD_HNO3       (time, ncol) float32 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    PRECC         (time, ncol) float32 dask.array<chunksize=(1, 174098), meta=np.ndarray>
    PRECL         (time, ncol) float32 dask.array<chunksize=(1, 174098), meta=np.ndarray>
Attributes:
    ne:                0
    np:                4
    Conventions:       CF-1.0
    source:            CAM
    case:              MUSICA_FCnudged30x8_20172018_082024
    logname:           demurray
    host:              derecho3
    initial_file:      /glade/derecho/scratch/demurray/inic/f.e22.FCnudged.ne...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/s...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [6]:
#Subset lat and lon for only over north america
musica_dim = musica_daily[['lat', 'lon']]
clipped_lat = musica_daily['lat'].clip(min=24, max=50)
clipped_lon = musica_daily['lon'].clip(min=235, max=294)

# Step 2: Update the dataset with clipped variables
musica_clipped = musica_dim.assign(lat=clipped_lat, lon=clipped_lon)
#musica_clipped

#Step 3. convert to a dataframe
mus_dim_df = musica_clipped.to_dataframe().reset_index()
mus_dim_df = mus_dim_df[['ncol', 'lat','lon']].drop_duplicates()
mus_dim_df = mus_dim_df.set_index('ncol')
mus_dim_df

,lat,lon
ncol,,
0,24.000000,235.000000
1,24.000000,235.000000
2,24.000000,235.375435
3,24.000000,235.764926
4,24.000000,235.000000
...,...,...
174093,47.263011,294.000000
174094,47.377896,294.000000
174095,47.216640,294.000000


In [7]:
#read in the dataframe associated with the lat lon of the NADP sites and DOC sites
pathData = '/glade/u/home/demurray/External File Uploads/NADP NTN wet dep data/'
os.chdir(pathData)
ntn = pd.read_csv('ntn.csv')
ntn['lon'] =  360 - (ntn['longitude'] * -1) #convert to same units as the model lon
ntn = ntn[['siteId', 'latitude', 'longitude', 'lon']]

#Also read in the DOC sites
pathData = '/glade/u/home/demurray/External File Uploads/DOC data/'
os.chdir(pathData)
doc_sites = pd.read_csv('DOC_wetdep_compiled.csv')
doc_sites = doc_sites[['siteId', 'latitude', 'longitude']].drop_duplicates()
doc_sites['lon'] =  360 - (doc_sites['longitude'] * -1) #convert to same units as the model lon

#merge the DOC and NTN site dfs together
all_sites = pd.concat([ntn, doc_sites], ignore_index=True)
all_sites.drop_duplicates()

#Clip site df to be the same range of lat and lon as the modelled data frame
all_sites['latitude'] = all_sites['latitude'].clip(lower=24, upper=50)
all_sites['lon'] = all_sites['lon'].clip(lower=235, upper=294)
all_sites_loc = all_sites[['siteId', 'latitude', 'lon']]
all_sites_loc

,siteId,latitude,lon
0,AB32,50.0000,248.3594
1,AB34,50.0000,248.8273
2,AB36,50.0000,248.9614
3,AK01,50.0000,235.0000
4,AK02,50.0000,235.0000
...,...,...,...
410,TN11,35.6645,276.4097
411,TX03,28.4667,262.2931
412,WA14,47.8597,236.0675
413,WV05,38.8794,279.1524


In [8]:
# Calculate distances between all points in mus_df and ntn_loc
distances = np.zeros((len(mus_dim_df), len(all_sites_loc)))

# Use tqdm to add a progress bar
for i, row in tqdm(mus_dim_df.iterrows(), total=len(mus_dim_df), desc='Calculating distances'):
    for j, row_ntn in all_sites_loc.iterrows():
        distances[i, j] = haversine(row['lat'], row['lon'], row_ntn['latitude'], row_ntn['lon'])

Calculating distances: 100%|██████████| 174098/174098 [1:22:33<00:00, 35.14it/s]


In [9]:
# Find the index of the closest point in mus_df for each point in ntn_loc
closest_indices = np.argmin(distances, axis=0)

# Keep only the latitude and longitude pairs in mus_df that correspond to the closest points in ntn_loc
mus_df_matched = mus_dim_df.iloc[closest_indices]

# Assign the corresponding 'siteId' from ntn_loc to mus_df_matched
mus_df_matched['siteId'] = all_sites_loc['siteId'].values
mus_df_matched = mus_df_matched.rename(columns = {'lat': 'lat_mod', 'lon' : 'lon_mod'})
mus_df_matched.reset_index(inplace = True) #checks -- there should be the same length of cells in mus_df_matched as ntn_loc

mus_ntn = pd.merge(mus_df_matched, all_sites_loc, on = 'siteId', how = 'left') #looks good!
mus_ntn

/glade/derecho/scratch/demurray/tmp/ipykernel_68123/3880182875.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mus_df_matched['siteId'] = all_sites_loc['siteId'].values


,ncol,lat_mod,lon_mod,siteId,latitude,lon
0,167072,50.000000,248.354064,AB32,50.0000,248.3594
1,51725,50.000000,248.828346,AB34,50.0000,248.8273
2,168858,50.000000,248.962827,AB36,50.0000,248.9614
3,3193,50.000000,235.000000,AK01,50.0000,235.0000
4,3193,50.000000,235.000000,AK02,50.0000,235.0000
...,...,...,...,...,...,...
454,166062,47.798792,236.020964,WA14,47.8597,236.0675
455,141507,38.937720,279.101951,WV05,38.8794,279.1524
456,141507,38.937720,279.101951,WV05,38.8794,279.1524
457,150484,44.877979,249.623190,WY08,44.9166,249.5797


In [10]:
#from here, make a dictionary of site id that corresponds with model lat and lon and then 
#subset the xarray by this dictionary before perfroming any dataframe manipulation
# Step 1: Subset the xarray.Dataset using ncol
subset_ds = musica_daily.sel(ncol=mus_df_matched['ncol'].values)

# Step 2: Assign siteId to the subset xarray.Dataset
# Convert DataFrame to xarray.Dataset for easier assignment
siteId_da = xr.DataArray(mus_df_matched['siteId'].values, dims=['ncol'], coords={'ncol': mus_df_matched['ncol'].values})
subset_ds = subset_ds.assign(siteId=siteId_da)
subset_ds
subset_ds.to_netcdf('/glade/u/home/demurray/Murray-NCAR-GVP/MUSICA Analyses/Data outputs/MUSICA_nadp_merged_gridded_daily_timeseries.nc')

In [11]:
subset_ds

<xarray.Dataset>
Dimensions:       (time: 729, ncol: 415)
Coordinates:
  * time          (time) datetime64[ns] 2017-01-01 2017-01-02 ... 2018-12-30
  * ncol          (ncol) int64 167072 51725 168858 3193 ... 166062 141507 150484
Data variables: (12/43)
    lat           (time, ncol) float64 dask.array<chunksize=(1, 415), meta=np.ndarray>
    lon           (time, ncol) float64 dask.array<chunksize=(1, 415), meta=np.ndarray>
    area          (time, ncol) float64 dask.array<chunksize=(1, 415), meta=np.ndarray>
    pom_a1SFWET   (time, ncol) float32 dask.array<chunksize=(1, 415), meta=np.ndarray>
    pom_a4SFWET   (time, ncol) float32 dask.array<chunksize=(1, 415), meta=np.ndarray>
    pom_c1SFWET   (time, ncol) float32 dask.array<chunksize=(1, 415), meta=np.ndarray>
    ...            ...
    WD_NH4        (time, ncol) float32 dask.array<chunksize=(1, 415), meta=np.ndarray>
    WD_NH3        (time, ncol) float32 dask.array<chunksize=(1, 415), meta=np.ndarray>
    WD_HNO3       (time, ncol) float32 dask.array<chunksize=(1, 415), meta=np.ndarray>
    PRECC         (time, ncol) float32 dask.array<chunksize=(1, 415), meta=np.ndarray>
    PRECL         (time, ncol) float32 dask.array<chunksize=(1, 415), meta=np.ndarray>
    siteId        (ncol) object 'AB32' 'AB34' 'AB36' ... 'WA14' 'WV05' 'WY08'
Attributes:
    ne:                0
    np:                4
    Conventions:       CF-1.0
    source:            CAM
    case:              MUSICA_FCnudged30x8_20172018_082024
    logname:           demurray
    host:              derecho3
    initial_file:      /glade/derecho/scratch/demurray/inic/f.e22.FCnudged.ne...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/s...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1